# Chapter 3: Data Cleaning and Preparation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aeleraqi/Python-for-Journalists/blob/main/notebooks/Chapter_03_Notebook.ipynb)

Companion notebook to *The Code Behind the Story: Data Skills with Python for Journalists*, second edition, by Amr Eleraqi. It holds every code cell of the chapter, in the book's order; the book explains each one. Run the cells from top to bottom.

## 3.1 Why cleaning comes first

### 3.1.1 pandas for cleaning

### 3.1.2 Dimensions of data quality

### 3.1.3 Common mistakes in cleaning

## 3.2 Tools and data for this chapter

In [ ]:
import pandas as pd
import numpy as np

### 3.2.1 Getting the data

In [ ]:
repo = ("https://raw.githubusercontent.com/"
        "aeleraqi/Python-for-Journalists/main")
!wget -q {repo}/data/Electric_Vehicle_Population_Data.csv

In [ ]:
ev_raw = pd.read_csv("Electric_Vehicle_Population_Data.csv")
ev = ev_raw.copy()

### 3.2.2 Where the data comes from

## 3.3 A first look at the table

### 3.3.1 Size and rows

In [ ]:
ev.shape

In [ ]:
pd.set_option("display.max_columns", 8)

In [ ]:
ev[["Make", "Model", "Model Year", "Electric Range",
    "Base MSRP"]].head(3)

In [ ]:
ev[["County", "City", "Make", "Model"]].sample(4, random_state=7)

### 3.3.2 Columns and types

In [ ]:
ev.columns

In [ ]:
short_name = {"Clean Alternative Fuel Vehicle (CAFV) Eligibility":
              "CAFV Eligibility"}
ev.rename(columns=short_name).info()

In [ ]:
ev.dtypes["Postal Code"]

### 3.3.3 Summary statistics

In [ ]:
ev[["Postal Code", "Model Year", "Electric Range", "Base MSRP",
    "Legislative District"]].describe()

In [ ]:
ev[["County", "City", "State", "Make", "Model"]].describe()

### 3.3.4 Counting values

In [ ]:
ev.nunique()

In [ ]:
ev["Electric Vehicle Type"].unique()

In [ ]:
ev["Make"].value_counts().head(10)

In [ ]:
(ev["Electric Vehicle Type"].value_counts(normalize=True) * 100).round(1)

## 3.4 Working with columns

### 3.4.1 Renaming columns

In [ ]:
ev.rename(columns={"Model Year": "model_year"}).columns[:6].tolist()

In [ ]:
ev.columns = (
    ev.columns
    .str.strip()                             # remove spaces at the ends
    .str.lower()                             # all lowercase
    .str.replace(" ", "_")                   # spaces become underscores
    .str.replace(r"[^\w]", "", regex=True)   # drop other characters
)
ev.columns

In [ ]:
ev = ev.rename(columns={
    "vin_110": "vin_1_10",
    "clean_alternative_fuel_vehicle_cafv_eligibility": "cafv_eligibility",
    "2020_census_tract": "census_tract_2020",
})
print(ev.columns.tolist())

### 3.4.2 Selecting columns

In [ ]:
ev.select_dtypes(include="number").columns

In [ ]:
ev.select_dtypes(exclude="number").columns

### 3.4.3 Dropping columns

In [ ]:
ev_small = ev.drop(columns=["vin_1_10", "electric_utility",
                            "vehicle_location", "census_tract_2020"])
ev_small.shape

### 3.4.4 Adding columns

In [ ]:
ev_small["source"] = "WA DOL via Kaggle"
ev_small["source"].head(2)

In [ ]:
ev_small["is_bev"] = (ev_small["electric_vehicle_type"]
                      == "Battery Electric Vehicle (BEV)")
ev_small["range_km"] = ev_small["electric_range"] * 1.609344
ev_small[["make", "model", "is_bev", "electric_range",
          "range_km"]].head(3)

In [ ]:
ev_small["is_bev"].sum()

### 3.4.5 Changing a column's type

In [ ]:
ev[["postal_code", "legislative_district", "census_tract_2020"]].head(3)

In [ ]:
ev.loc[ev["state"] == "MA", ["city", "state", "postal_code"]]

In [ ]:
# This cell stops with an error on purpose: read the message.
ev["postal_code"].astype(int)

In [ ]:
ev["postal_code"] = (
    ev["postal_code"]
    .astype("Int64")      # whole numbers that allow missing values
    .astype("string")     # then text
    .str.zfill(5)         # pad with zeros on the left to five characters
)
ev.loc[ev["state"] == "MA", ["city", "state", "postal_code"]]

In [ ]:
ev["census_tract_2020"] = (
    ev["census_tract_2020"].astype("Int64").astype("string").str.zfill(11)
)
ev["legislative_district"] = ev["legislative_district"].astype("Int64")
ev[["postal_code", "legislative_district", "census_tract_2020"]].head(3)

In [ ]:
ev.loc[ev["state"] == "AL", ["city", "state", "census_tract_2020"]]

In [ ]:
prices = pd.Series(["69900", "31,950", "n/a"])
pd.to_numeric(prices, errors="coerce")

## 3.5 Cleaning text

### 3.5.1 Capital letters

In [ ]:
print(ev["city"].nunique())
print(ev["city"].str.upper().nunique())

In [ ]:
spellings = ev["city"].dropna().drop_duplicates()
spellings[spellings.str.upper().duplicated(keep=False)]

In [ ]:
mccleary = ev["city"].isin(["Mccleary", "McCleary"])
ev.loc[mccleary, ["city", "postal_code"]].value_counts()

In [ ]:
ev["city"] = ev["city"].replace({"Mccleary": "McCleary"})
ev["city"].nunique()

In [ ]:
ev["make"].str.title().value_counts().head(6)

### 3.5.2 Spaces at the ends

In [ ]:
for col in ["county", "city", "state", "make", "model"]:
    values = ev[col].dropna()
    print(col, (values != values.str.strip()).sum())

In [ ]:
typed = pd.Series(["Seattle", "Seattle ", " seattle"])
print(typed.value_counts())
print(typed.str.strip().str.title().value_counts())

### 3.5.3 Removing characters with a regular expression

In [ ]:
models = pd.Series(ev["model"].unique())
stripped = models.str.replace(r"[^a-zA-Z0-9 ]", "", regex=True)
changes = pd.DataFrame({"before": models, "after": stripped})
changes[changes["before"] != changes["after"]].head(8)

In [ ]:
ev["ev_type"] = ev["electric_vehicle_type"].str.extract(
    r"\((\w+)\)", expand=False)
ev["ev_type"].value_counts()

### 3.5.4 Splitting the location into longitude and latitude

In [ ]:
ev["vehicle_location"].head(3)

In [ ]:
coordinates = ev["vehicle_location"].str.extract(
    r"POINT \((?P<longitude>\S+) (?P<latitude>\S+)\)")
ev["longitude"] = coordinates["longitude"].astype(float)
ev["latitude"] = coordinates["latitude"].astype(float)
ev[["vehicle_location", "longitude", "latitude"]].head(3)

In [ ]:
# This cell stops with an error on purpose: read the message.
def extract_lat_long(point_str):
    point_str = point_str.replace("POINT (", "").replace(")", "")
    longitude, latitude = map(float, point_str.split())
    return latitude, longitude

ev["vehicle_location"].apply(extract_lat_long)

## 3.6 Missing values

### 3.6.1 Counting missing values

In [ ]:
missing = ev.isna().sum()
missing[missing > 0]

### 3.6.2 Ask why a value is missing

In [ ]:
no_district = ev["legislative_district"].isna()
ev.loc[no_district, "state"].value_counts().head()

In [ ]:
print((ev["state"] != "WA").sum())
print((no_district & (ev["state"] == "WA")).sum())

In [ ]:
ev.loc[ev["county"].isna(), ["state", "make", "model", "model_year"]]

### 3.6.3 When zero means "unknown"

In [ ]:
ev["electric_range"].value_counts().head()

In [ ]:
zero_range = ev["electric_range"] == 0
ev.loc[zero_range, "cafv_eligibility"].value_counts()

In [ ]:
import matplotlib.pyplot as plt

share_zero = zero_range.groupby(ev["model_year"]).mean() * 100
share_zero = share_zero.loc[2011:]

fig, ax = plt.subplots(figsize=(5.4, 2.8))
share_zero.plot.bar(ax=ax)
ax.set_title("Electric range recorded as 0, by model year")
ax.set_xlabel("Model year")
ax.set_ylabel("Share of vehicles (%)")
plt.show()

In [ ]:
has_price = ev["base_msrp"] > 0
print((ev["base_msrp"] == 0).sum())
print(has_price.sum())
print(ev.loc[has_price, "model_year"].between(2012, 2019).sum())

In [ ]:
print(ev["electric_range"].mean())
print(ev.loc[ev["electric_range"] > 0, "electric_range"].mean())

In [ ]:
ev["electric_range"] = ev["electric_range"].replace(0, np.nan)
ev["base_msrp"] = ev["base_msrp"].replace(0, np.nan)

print(ev["electric_range"].count(), ev["electric_range"].mean())
print(ev["base_msrp"].count(), ev["base_msrp"].median())

### 3.6.4 Dropping rows with missing values

In [ ]:
print(len(ev))
print(len(ev.dropna()))

In [ ]:
by_county = ev.dropna(subset=["county"])
len(ev) - len(by_county)

### 3.6.5 Filling missing values

In [ ]:
county_counts = ev["county"].fillna("Unknown").value_counts()
print(county_counts["Unknown"])

In [ ]:
by_model = ev.groupby("model")["legislative_district"]
model_average = by_model.transform("mean")
model_average[no_district].head()

In [ ]:
practice = ev[["county"]].copy()
practice["county"].fillna("Unknown", inplace=True)
print(practice["county"].isna().sum())

In [ ]:
practice["county"] = practice["county"].fillna("Unknown")
print(practice["county"].isna().sum())

## 3.7 Duplicates

### 3.7.1 What counts as a duplicate here

In [ ]:
print(ev["dol_vehicle_id"].is_unique)
print(ev.duplicated().sum())

### 3.7.2 Rows that look alike

In [ ]:
other_columns = ev.columns.drop("dol_vehicle_id")
print(ev.duplicated(subset=other_columns).sum())

In [ ]:
look_alike = ev.duplicated(subset=other_columns, keep=False)
(ev.loc[look_alike, ["dol_vehicle_id", "make", "model", "model_year",
                     "census_tract_2020"]]
 .sort_values(["census_tract_2020", "model", "model_year"])
 .head(4))

In [ ]:
len(ev) - len(ev.drop_duplicates(subset=other_columns))

In [ ]:
print(ev["vin_1_10"].duplicated().sum())
ev["vin_1_10"].value_counts().head(3)

## 3.8 Dates

### 3.8.1 A year is not a date

In [ ]:
ev["model_year"].value_counts().sort_index().tail(4)

In [ ]:
model_year_date = pd.to_datetime(ev["model_year"], format="%Y")
model_year_date.head(3)

In [ ]:
model_year_date.dt.year.head(3)

### 3.8.2 Dates written as text

In [ ]:
monthly = pd.DataFrame({
    "date": ["2023-07-31T00:00:00.000", "2023-08-31T00:00:00.000",
             "2023-09-30T00:00:00.000", "2023-10-31T00:00:00.000"],
    "ev_total": [143260, 149672, 154182, 159358],
})
monthly.dtypes

In [ ]:
monthly["date"] = pd.to_datetime(monthly["date"])
monthly.dtypes

In [ ]:
monthly["year"] = monthly["date"].dt.year
monthly["month"] = monthly["date"].dt.month
monthly["month_name"] = monthly["date"].dt.month_name()
monthly["day"] = monthly["date"].dt.day
monthly

In [ ]:
monthly.sort_values("date", ascending=False)[["date", "ev_total"]]

In [ ]:
monthly[["date", "ev_total"]].assign(our_file=len(ev))

### 3.8.3 Layouts and errors

In [ ]:
typed_dates = pd.Series(["2023-09-30", "30/09/2023",
                         "September 30, 2023", "not known"])

In [ ]:
# This cell stops with an error on purpose: read the message.
pd.to_datetime(typed_dates, format="%Y-%m-%d")

In [ ]:
converted = pd.to_datetime(typed_dates, format="%Y-%m-%d", errors="coerce")
print(converted)
print(converted.isna().sum())

In [ ]:
print(pd.to_datetime("30/09/2023", format="%d/%m/%Y"))
print(pd.to_datetime("September 30, 2023", format="%B %d, %Y"))

## 3.9 Hands-on exercises

### Exercise 1: The Seattle fleet

In [ ]:
seattle = ev[ev["city"] == "Seattle"]
print(len(seattle))
seattle["make"].value_counts().head()

### Exercise 2: Plug-in hybrids in Spokane

In [ ]:
spokane = ev[ev["county"] == "Spokane"]
spokane_share = spokane["ev_type"].value_counts(normalize=True)
state_share = ev["ev_type"].value_counts(normalize=True)
print((spokane_share * 100).round(1))
print((state_share * 100).round(1))

### Exercise 3: Recorded prices

In [ ]:
priced = ev[ev["base_msrp"].notna()]
print(len(priced))
print(priced["model"].value_counts().head(3))
print(priced["base_msrp"].median())

### Exercise 4: The main utility

In [ ]:
ev["main_utility"] = ev["electric_utility"].str.split("|").str[0]
ev["main_utility"].value_counts().head()

### Exercise 5: Saving and reading back

In [ ]:
ev.to_csv("ev_cleaned.csv", index=False)

check = pd.read_csv("ev_cleaned.csv")
print(check.loc[check["state"] == "MA", "postal_code"].tolist())

check = pd.read_csv("ev_cleaned.csv",
                    dtype={"postal_code": "string",
                           "census_tract_2020": "string"})
print(check.loc[check["state"] == "MA", "postal_code"].tolist())

### A challenge that combines the steps

In [ ]:
older_bev = ev[(ev["ev_type"] == "BEV") & (ev["model_year"] <= 2020)]
recorded = older_bev[older_bev["electric_range"].notna()]
top_make = recorded["make"].value_counts().index[0]
print(top_make)
top_make_cars = recorded[recorded["make"] == top_make]
print(top_make_cars["electric_range"].median())

## 3.10 Project: Washington's electric cars

### Step 1: Load the data and keep the original

In [ ]:
import pandas as pd
import numpy as np

raw = pd.read_csv("Electric_Vehicle_Population_Data.csv")
ev = raw.copy()
ev.shape

### Step 2: Clean the column names

In [ ]:
ev.columns = (
    ev.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace(r"[^\w]", "", regex=True)
)
ev = ev.rename(columns={
    "vin_110": "vin_1_10",
    "clean_alternative_fuel_vehicle_cafv_eligibility": "cafv_eligibility",
    "2020_census_tract": "census_tract_2020",
})

### Step 3: Store codes as codes

In [ ]:
ev["postal_code"] = (ev["postal_code"].astype("Int64")
                     .astype("string").str.zfill(5))
ev["census_tract_2020"] = (ev["census_tract_2020"].astype("Int64")
                           .astype("string").str.zfill(11))
ev["legislative_district"] = ev["legislative_district"].astype("Int64")

### Step 4: Tidy the text

In [ ]:
ev["city"] = ev["city"].replace({"Mccleary": "McCleary"})
ev["ev_type"] = ev["electric_vehicle_type"].str.extract(
    r"\((\w+)\)", expand=False)

coordinates = ev["vehicle_location"].str.extract(
    r"POINT \((?P<longitude>\S+) (?P<latitude>\S+)\)")
ev["longitude"] = coordinates["longitude"].astype(float)
ev["latitude"] = coordinates["latitude"].astype(float)

### Step 5: Mark the unknown values

In [ ]:
print("zero ranges:", (ev["electric_range"] == 0).sum())
print("zero prices:", (ev["base_msrp"] == 0).sum())

ev["electric_range"] = ev["electric_range"].replace(0, np.nan)
ev["base_msrp"] = ev["base_msrp"].replace(0, np.nan)

### Step 6: Check for duplicates

In [ ]:
print("IDs unique:", ev["dol_vehicle_id"].is_unique)
print("identical rows:", ev.duplicated().sum())

### Step 7: Save the cleaned table

In [ ]:
ev.to_csv("ev_clean.csv", index=False)
print(len(raw), len(ev))

### Step 8: Report

In [ ]:
make_share = (ev["make"].value_counts(normalize=True) * 100).round(1)
make_share.head(5)

In [ ]:
import matplotlib.pyplot as plt

top_makes = ev["make"].value_counts().head(10).sort_values()
labels = top_makes.index.str.title().str.replace("Bmw", "BMW")

fig, ax = plt.subplots(figsize=(5.4, 3.2))
ax.barh(labels, top_makes.values)
ax.set_title("Electric vehicles registered in Washington, top ten makes")
ax.set_xlabel("Number of vehicles")
ax.set_ylabel("Make")
plt.show()

In [ ]:
county_share = (ev["county"].value_counts(normalize=True) * 100).round(1)
county_share.head(5)

In [ ]:
ev["ev_type"].value_counts()

### Step 9: The caveats

## 3.11 Habits worth keeping

## Solutions to the "Your turn" exercises

### Section 3.3: A first look at the table

In [ ]:
# 1
print((raw["Model Year"] == 2023).sum())

# 2
print(raw[["City", "Make", "Model"]].sample(5, random_state=1))

# 3
print(raw["City"].nunique())
print((raw["City"] == "Spokane").sum())

# 4
print(round((raw["County"] == "King").mean() * 100, 1))

# 5
print(raw["Model"].value_counts().head(10))

### Section 3.4: Working with columns

In [ ]:
# 1
renamed = raw.rename(columns={"Model Year": "year", "Make": "brand"})
print(renamed.columns[:6].tolist())

# 2
places = ev[["county", "city", "state", "postal_code"]]
print(places.head(3))

# 3
ev_small = ev.drop(columns=["vin_1_10", "electric_utility",
                            "vehicle_location", "census_tract_2020"])
ev_small["source"] = "WA DOL via Kaggle"
ev_small["range_km"] = ev_small["electric_range"] * 1.609344
print(ev_small.drop(columns=["source", "range_km"]).shape)

# 4
ev_small["age_in_2023"] = 2023 - ev_small["model_year"]
print(ev_small["age_in_2023"].max())

# 5
nj = ev.loc[ev["state"] == "NJ", "postal_code"]
print(sorted(set(nj)))
print((nj.str.len() == 5).all())

### Section 3.5: Cleaning text

In [ ]:
# 1
counties = ev["county"].dropna().drop_duplicates()
print(counties[counties.str.upper().duplicated(keep=False)])

# 2
models = pd.Series(ev["model"].unique())
print(models.str.contains("-").sum())

# 3
ev["main_utility"] = ev["electric_utility"].str.split("|").str[0]
print(ev["main_utility"].value_counts().head())

# 4
print(ev.nsmallest(3, "latitude")[["city", "state", "latitude",
                                   "longitude"]])

# 5
print(ev["ev_type"].str.lower().value_counts())

### Section 3.6: Missing values

In [ ]:
# 1
missing_raw = raw.isna().sum()
print(missing_raw[missing_raw > 0])

# 2
phev = ev[ev["ev_type"] == "PHEV"]
bev = ev[ev["ev_type"] == "BEV"]
print(round(phev["electric_range"].notna().mean() * 100, 1))
print(round(bev["electric_range"].notna().mean() * 100, 1))

# 3
bev_2020 = bev[bev["model_year"] == 2020]
print(bev_2020["electric_range"].median())

# 4
no_location = ev["vehicle_location"].isna() & ev["postal_code"].notna()
print(ev.loc[no_location, ["city", "county", "postal_code"]])

# 5
districts = ev["legislative_district"].value_counts(dropna=False)
print(districts.sort_index().tail(3))

### Section 3.7: Duplicates

In [ ]:
# 1
print(ev.duplicated(subset=["make", "model", "model_year"]).sum())

# 2
tesla = ev[ev["make"] == "TESLA"]
print(tesla["vin_1_10"].nunique())

# 3
repeated = ev[ev["dol_vehicle_id"].duplicated(keep=False)]
print(len(repeated))

# 4
pairs = ev[["make", "model"]].drop_duplicates()
print(len(pairs))

### Section 3.8: Dates

In [ ]:
# 1
dates = pd.to_datetime(ev["model_year"].head(), format="%Y")
print(dates.dt.year.tolist(), dates.dt.month.tolist())

# 2
monthly["change"] = monthly["ev_total"].diff()
print(monthly[["date", "ev_total", "change"]])

# 3
print(pd.to_datetime("2023-12-31T00:00:00.000").day_name())

# 4
print(pd.to_datetime(pd.Series(["31.08.2023", "30.09.2023", "unknown"]),
                     format="%d.%m.%Y", errors="coerce"))